# CA Experiment 6 — From-Scratch ~80M ADA Daily-QA Agent (Persona-Bearing, QPM-in-scope)

**Runtime:** Colab Pro **A100/L4 GPU** + bounded Anthropic API (Sonnet persona/style/refusal data + Sonnet 4.5 judge) + free open corpora.

One from-scratch ~80M decoder-only model as ADA's **daily-QA** agent: reads retrieved context → answers/abstains **in character**. Persona/SCI/SMC are trained, and the **QPM output is compiled into the weights** via a `persona_state` `<|persona|>` channel (plan §5.5).

| Axis | In scope | Measured by |
|---|---|---|
| Grounded-QA competence (H1) | ✅ | correct-and-grounded % + SQuAD2 EM/F1 |
| Calibrated abstention / SMC-C (H2) | ✅ | abstention F1 + hallucination rate |
| Persona T/E/C/S (H3) | ✅ | Exp 1–5 PersonaScore harness (Sonnet judge) |
| SCI-refresh policy (H4) | ✅ | R0 vs R1 (refresh @15/30) |
| Fallback gap (H5/RQ5) | ✅ | vs fine-tuned SmolLM2-135M |
| **QPM-as-weight-supervision (RQ6)** | ✅ | QPM-on vs `--no-qpm` ablation |

The notebook is **resumable**: checkpoints mirror to Drive; re-running a training cell resumes with `--resume`. Timeline: **P0** provision → **P1** tokenizer + pilot gate → **P2** Stage-A pretrain → **P3** Stage-B SFT → **P4** baseline → **P5** eval → **P6** decision.

## Cell 1: Mount Drive, set paths, load API key

Exp 6 is **self-contained** (no cross-experiment assets). The Anthropic key is only needed for the Sonnet persona-data and judging cells — all training/corpus steps are free.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
from pathlib import Path

PROJECT_DIR = '/content/drive/MyDrive/CA_Experiment_6'
CKPT_DIR    = f'{PROJECT_DIR}/checkpoints'   # mirrored to Drive (survives disconnects)
DATA_DIR    = f'{PROJECT_DIR}/data'
RESULTS_DIR = f'{PROJECT_DIR}/results'

# Pretrain bins live on LOCAL Colab SSD for training (random-access memmap reads over the
# Drive FUSE mount are far too slow and stall Stage-A). A one-time sequential copy to the
# Drive archive below keeps a durable, downloadable copy — see cell P1b.
LOCAL_PT    = '/content/data/pretrain'       # fast local disk (ephemeral)
DRIVE_PT    = f'{DATA_DIR}/pretrain'         # durable archive on Drive (persistent, downloadable)

assert os.path.exists(PROJECT_DIR), f'Upload CA_Experiment_6 to Drive first! Not found at {PROJECT_DIR}'
for d in (CKPT_DIR, DRIVE_PT, f'{DATA_DIR}/persona_scripts', RESULTS_DIR, LOCAL_PT):
    os.makedirs(d, exist_ok=True)


def _parse_env(path):
    """Tolerant .env parser: handles BOM, blank/comment lines, `export `, quotes, spaces."""
    vals = {}
    for raw in open(path, encoding='utf-8-sig'):
        line = raw.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        if line.startswith('export '):
            line = line[len('export '):]
        k, v = line.split('=', 1)
        vals[k.strip()] = v.strip().strip('"').strip("'")
    return vals


# API key — Colab Secrets preferred, then a .env on Drive (same pattern as Exp 4/5).
ANTHROPIC_API_KEY = ''
for secret_name in ('CHA_EXPERIMENT_SONNET_KEY', 'ANTHROPIC_API_KEY'):
    try:
        ANTHROPIC_API_KEY = userdata.get(secret_name)
        if ANTHROPIC_API_KEY:
            print(f'API key loaded from Colab Secrets ({secret_name})')
            break
    except Exception:
        pass
if not ANTHROPIC_API_KEY:
    env_path = os.path.join(PROJECT_DIR, '.env')
    if os.path.exists(env_path):
        env = _parse_env(env_path)
        ANTHROPIC_API_KEY = (env.get('CHA_EXPERIMENT_SONNET_KEY')
                             or env.get('ANTHROPIC_API_KEY') or '')
        if ANTHROPIC_API_KEY:
            print(f'API key loaded from {env_path}')
        else:
            print(f'{env_path} found, but no CHA_EXPERIMENT_SONNET_KEY / ANTHROPIC_API_KEY '
                  f'in it. Keys present: {list(env)}')
    else:
        print(f'No .env at {env_path}. Folder contents: {sorted(os.listdir(PROJECT_DIR))[:25]}')
if ANTHROPIC_API_KEY:
    os.environ['CHA_EXPERIMENT_SONNET_KEY'] = ANTHROPIC_API_KEY
    os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY

print('='*64)
print(f'Project dir : {PROJECT_DIR}')
print(f'Checkpoints : {CKPT_DIR}   (Drive — model .pt files, downloadable)')
print(f'Pretrain    : {LOCAL_PT} (local, fast)  ↔  {DRIVE_PT} (Drive archive)')
print(f'API key     : {("..."+ANTHROPIC_API_KEY[-8:]) if ANTHROPIC_API_KEY else "NOT SET (Sonnet cells will be skipped)"}')
print('='*64)

## Cell 2: Install dependencies & load Exp-6 assets

GPU torch (preinstalled on Colab) + tokenizers + datasets (free corpora) + anthropic (persona data + judge) + **qiskit/qiskit-aer + vaderSentiment** (the QPM, plan §5.5).

In [ ]:
!pip install -q -U tokenizers datasets anthropic python-dotenv qiskit qiskit-aer vaderSentiment tqdm

import os, sys
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

import ca_assets as A
import qpm_bridge as QB
from model import ADA_80M, ADA_PILOT

print('special tokens :', A.SPECIAL_TOKENS)
print('dimensions     :', A.DIMENSIONS, '| probe turns', A.PROBE_TURNS)
print('ADA profile    :', QB.ADA_PROFILE)
_ps = QB.build_persona_state([QB.extract_d_vector('explain planck constant, I am unsure')])
print('QPM sanity     : source={source} register={register} ambivalence={ambivalence} certainty={certainty}'.format(**_ps))
print('-'*64)
print(f'Model (80M)    : d_model={ADA_80M.d_model} layers={ADA_80M.n_layers} heads={ADA_80M.n_heads} '
      f'vocab={ADA_80M.vocab_size} ctx={ADA_80M.max_seq_len} ffn={ADA_80M.hidden_dim()} head_dim={ADA_80M.head_dim()}')
print(f'Pilot          : d_model={ADA_PILOT.d_model} layers={ADA_PILOT.n_layers} (§5.4 gate)')

## Cell 3: Verify GPU & report throughput expectations

In [ ]:
import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Change runtime type to a GPU (A100/L4 for the full run; T4 = smoke only).')

p = torch.cuda.get_device_properties(0)
vram = p.total_memory / 1e9
print(f'GPU            : {p.name}')
print(f'VRAM           : {vram:.1f} GB')
print(f'bf16 supported : {torch.cuda.is_bf16_supported()}')
print(f'compute cap    : {p.major}.{p.minor}')

# Rough Stage-A ETA for 2B tokens by GPU class (plan §5.3).
name = p.name.lower()
if 'a100' in name:      eta = '~11 h (~50k tok/s)'
elif 'l4' in name:      eta = '~20-25 h (~25k tok/s)'
elif 't4' in name:      eta = '~50 h — SMOKE/PILOT ONLY (§5.3)'
else:                   eta = 'unknown — benchmark with the throughput logs'
print(f'Stage-A 2B tok : {eta}')
if vram < 16:
    print('WARNING: <16 GB VRAM — reduce --batch-size / --seq-len; the 80M model itself fits, but keep an eye on OOM.')
print('Exp 6 GPU environment: READY')

## Cell 4: Anthropic API smoke test

Confirms the key works before the persona-data / judging cells. Skipped (not fatal) if no key is set.

In [ ]:
GEN_MODEL   = 'claude-sonnet-4-6'   # persona/style/refusal data (plan §4.2); $3/$15 per 1M tok
JUDGE_MODEL = 'claude-sonnet-4-5'   # judge id matches Exp 1-5; $3/$15 per 1M tok

if os.environ.get('CHA_EXPERIMENT_SONNET_KEY'):
    import anthropic
    _client = anthropic.Anthropic()
    _r = _client.messages.create(model=JUDGE_MODEL, max_tokens=10,
                                 messages=[{'role': 'user', 'content': 'Say "ok".'}])
    print(f'API smoke test : {_r.content[0].text.strip()}  (gen={GEN_MODEL}, judge={JUDGE_MODEL})')
else:
    print('No API key — the Sonnet persona-data (P1d) and judging (P5) cells will be skipped.')

---
## P0 · Provision — free Stage-A corpus text (for the tokenizer)

Stream a slice of FineWeb-Edu (or offline Wikipedia/ZIM) to a text file the BPE trains on. FREE (plan §4.2). Resumable — skips if the sample exists.

In [ ]:
from datasets import load_dataset
from tqdm import tqdm
import time

SAMPLE_TXT     = 'data/pretrain/corpus_sample.txt'
TOKENIZER_DOCS = 200_000   # docs used to TRAIN the BPE (raise for a richer vocab)

if not Path(SAMPLE_TXT).exists():
    t0 = time.time()
    ds = load_dataset('HuggingFaceFW/fineweb-edu', name='sample-10BT', split='train', streaming=True)
    n_chars = 0
    with open(SAMPLE_TXT, 'w') as f:
        for i, ex in enumerate(tqdm(ds, total=TOKENIZER_DOCS, desc='corpus sample')):
            if i >= TOKENIZER_DOCS:
                break
            txt = ex['text'].replace('\n', ' ')
            n_chars += len(txt)
            f.write(txt + '\n')
    print(f'wrote {SAMPLE_TXT}: {TOKENIZER_DOCS:,} docs, {n_chars/1e6:.1f} M chars in {time.time()-t0:.0f}s')
else:
    print(f'{SAMPLE_TXT} exists ({os.path.getsize(SAMPLE_TXT)/1e6:.1f} MB) — skipping')

## P1a · Train the 16k BPE tokenizer (+ ADA/QPM special tokens)

In [ ]:
if not Path('tokenizer/ada_bpe.json').exists():
    !python train_tokenizer.py --input data/pretrain/corpus_sample.txt --vocab-size 16000 --out tokenizer/ada_bpe.json
else:
    print('tokenizer exists — skipping')

from tokenizer_util import ADATokenizer
tok = ADATokenizer.load()
print(f'tokenizer      : vocab={tok.vocab_size}  persona_id={tok.persona_id}  assistant_id={tok.assistant_id}  eot_id={tok.eot_id}')
_ids = tok.encode('Tungsten melts at 3422 C.')
print(f'roundtrip test : {_ids[:8]}... -> {tok.decode(_ids)!r}')

## P1b · Tokenize Stage-A corpus → uint16 bins (local disk + Drive archive)

Target 1–3 B tokens (plan §5.2). Bins are built on **local SSD** (`LOCAL_PT`) for fast training, then copied **once** to the **Drive archive** (`DRIVE_PT`) so a durable, downloadable copy survives disconnects. On reconnect this cell restores from the archive instead of re-tokenizing. The model weights are separate — they live as `.pt` checkpoints under `CKPT_DIR` on Drive.

In [ ]:
import shutil

def _have(d):
    return os.path.exists(f'{d}/train.bin') and os.path.exists(f'{d}/val.bin')

if _have(LOCAL_PT):
    print(f'local pretrain bins present in {LOCAL_PT} — using them')
elif _have(DRIVE_PT):
    print(f'restoring pretrain bins from Drive archive → {LOCAL_PT} (sequential copy)...')
    for f in ('train.bin', 'val.bin'):
        shutil.copy(f'{DRIVE_PT}/{f}', f'{LOCAL_PT}/{f}')
    print('restored from archive (no re-tokenization needed)')
else:
    # Build on local disk (fast), reading the tokenizer from Drive.
    !python prepare_data.py pretrain --hf-dataset HuggingFaceFW/fineweb-edu --hf-config sample-10BT \
        --text-field text --out-dir $LOCAL_PT --max-tokens 2000000000
    print(f'archiving bins to Drive → {DRIVE_PT} (one-time sequential copy, downloadable) ...')
    for f in ('train.bin', 'val.bin'):
        shutil.copy(f'{LOCAL_PT}/{f}', f'{DRIVE_PT}/{f}')
    print('archived to Drive')

import numpy as np
for split in ('train', 'val'):
    b = Path(f'{LOCAL_PT}/{split}.bin')
    if b.exists():
        print(f'{split}.bin : {b.stat().st_size//2:,} tokens ({b.stat().st_size/1e6:.0f} MB, local)')

## P1c · Free reading-skill QA + held-out eval sets

SQuAD2 backbone (unanswerables supervise H2 abstention) → §4.3 records. Eval sets carry gold answers for EM/F1. FREE.

In [ ]:
if not Path('data/qa_sft.jsonl').exists():
    !python prepare_data.py qa-sft --out data/qa_sft.jsonl        # SQuAD2 train → reading skill
if not Path('data/eval_answerable.jsonl').exists():
    !python prepare_data.py eval --n 200                          # held-out answerable + unanswerable

import json, collections
cnt = collections.Counter(json.loads(l)['source'] for l in open('data/qa_sft.jsonl'))
print('qa_sft.jsonl by source :', dict(cnt), '| total', sum(cnt.values()))
print('eval_answerable        :', sum(1 for _ in open('data/eval_answerable.jsonl')))
print('eval_unanswerable      :', sum(1 for _ in open('data/eval_unanswerable.jsonl')))

## P1d · ADA persona layer (Sonnet, bounded) — QPM-conditioned

Each persona/style/refusal record is tagged with a QPM `persona_state` (marginals + valence + register + ambivalence/certainty) from the user turns' d-vectors; the affect directive is fed to the teacher so the target reflects it (plan §5.5). Bounded by `--budget`; raw archived to `data/raw_sonnet/`.

**This spends API budget** (~$25–60, plan §7.1). Lower the budgets or skip to economize.

In [ ]:
if os.environ.get('CHA_EXPERIMENT_SONNET_KEY'):
    # Persona conversations (multi-turn, episodic callbacks + capability-edge abstention)
    !python gen_persona_data.py persona --model $GEN_MODEL --budget 40 --out data/qa_sft.jsonl
    # ADA-voice restyling of a slice of the free QA answers
    !python gen_persona_data.py style   --model $GEN_MODEL --budget 40 --inputs data/qa_sft.jsonl --out data/qa_sft.jsonl
    # Refusal phrasing set (near-miss / off-topic / empty context)
    !python gen_persona_data.py refusal --model $GEN_MODEL --budget 30 --out data/qa_sft.jsonl
    # ~20 PersonaScore eval conversation scripts (40 user turns each)
    !python gen_persona_data.py eval-scripts --model $GEN_MODEL --n-scripts 20 --n-turns 40
    import collections, json
    cnt = collections.Counter(json.loads(l)['source'] for l in open('data/qa_sft.jsonl'))
    print('qa_sft by source :', dict(cnt))
    print('persona scripts  :', len(list(Path('data/persona_scripts').glob('script_*.json'))))
else:
    print('No API key — skipping Sonnet persona data. (Free reading-skill SFT still trains persona-thin.)')

---
## Pilot gate (plan §5.4) — does the pipeline learn at all?

Tiny model (d_model 256, 6 layers) on a small slice + the SFT data, on any GPU. Confirm loss falls and the assistant span is learned **before** spending A100 hours. Detailed loss/throughput logs stream from the training scripts.

In [ ]:
!python train_pretrain.py --config pilot --train-bin $LOCAL_PT/train.bin --val-bin $LOCAL_PT/val.bin \
    --ckpt-dir $CKPT_DIR --max-steps 2000 --batch-size 16 --grad-accum 4 --seq-len 512 --ckpt-interval 500
!python train_sft.py --init $CKPT_DIR/pretrain_pilot_final.pt --sft data/qa_sft.jsonl \
    --pretrain-bin $LOCAL_PT/train.bin --ckpt-dir $CKPT_DIR --max-steps 800 --seq-len 512

In [ ]:
# Pilot eyeball: grounded answer + abstention + QPM persona channel live (plan §5.4 gate)
from evaluate import ADAGenerator
g = ADAGenerator(f'{CKPT_DIR}/sft_final.pt', 'tokenizer/ada_bpe.json')
ctx = 'Tungsten has a melting point of 3422 degrees Celsius, the highest of all metals.'
print('GROUNDED  Q: melting point of tungsten?')
print('          A:', g.generate('What is the melting point of tungsten?', context=ctx))
print('ABSTAIN   Q: boiling point of nitrogen? (not in context)')
print('          A:', g.generate('What is the boiling point of nitrogen?', context=ctx))
print('\nPilot gate: PASS if loss fell, grounded answer is on-topic, and abstention fires. Only then run P2.')

---
## P2 · Stage-A pretrain (~80M, A100)

Next-token LM, cosine LR + warmup, bf16, checkpoint-to-Drive every 500 steps. **Resumable** — re-run this cell with `--resume` after a disconnect. Stops on val-loss plateau. Watch the `tok/s` and `val_loss` logs.

In [ ]:
!python train_pretrain.py --config 80m --train-bin $LOCAL_PT/train.bin --val-bin $LOCAL_PT/val.bin \
    --ckpt-dir $CKPT_DIR --max-steps 40000 --batch-size 32 --grad-accum 8 --seq-len 1024 \
    --ckpt-interval 500 --resume

import glob
print('Stage-A checkpoints on Drive:')
for f in sorted(glob.glob(f'{CKPT_DIR}/pretrain_80m*.pt'))[-5:]:
    print(f'  {os.path.basename(f)}  ({os.path.getsize(f)/1e6:.0f} MB)')

## P3 · Stage-B SFT (grounded-QA + ADA persona/style/refusal + QPM channel)

Assistant-span-masked loss; Stage-A replay to prevent fluency forgetting; the `<|persona|>` QPM channel is present in persona-bearing examples.

In [ ]:
!python train_sft.py --init $CKPT_DIR/pretrain_80m_best.pt --sft data/qa_sft.jsonl \
    --pretrain-bin $LOCAL_PT/train.bin --ckpt-dir $CKPT_DIR \
    --max-steps 6000 --batch-size 16 --grad-accum 4 --seq-len 1024 --replay-frac 0.1 --resume
print('SFT model:', f'{CKPT_DIR}/sft_final.pt', '(', os.path.getsize(f'{CKPT_DIR}/sft_final.pt')/1e6, 'MB )')

## P4 · Fallback baseline (RQ5 / H5, optional)

Fine-tune a small **pretrained** base (SmolLM2-135M) on the identical Stage-B data so the fallback is a one-step swap. Run only if you want the H5 gap now; skip to save compute. *(Uses `transformers`; add it to the install cell.)*

In [ ]:
# Placeholder for the RQ5 baseline: fine-tune HuggingFaceTB/SmolLM2-135M on data/qa_sft.jsonl
# (reformat records to the base's chat template; run §6.1-6.2 on it). The corpus/pipeline are
# reused unchanged — only the base swaps. Left as a documented optional step (plan §6.4).
print('RQ5 fallback baseline: optional — see plan §6.4 / decision-rule H1-fail row.')

---
## P5 · Evaluation

### QA competence — H1 (correct-and-grounded + EM/F1) and H2 (abstention F1 + hallucination)

In [ ]:
!python evaluate.py qa --checkpoint $CKPT_DIR/sft_final.pt \
    --answerable data/eval_answerable.jsonl --unanswerable data/eval_unanswerable.jsonl \
    --out results/qa_results.json

import json
m = json.load(open('results/qa_results.json'))['metrics']
print('='*56)
print(f"H1 correct-and-grounded : {m['H1_correct_grounded_rate']:.3f}  (>=0.70 -> {m['H1_pass']})")
print(f"   SQuAD2 EM / F1        : {m['squad_EM']} / {m['squad_F1']}")
print(f"H2 abstention F1        : {m['H2_abstention_F1']:.3f}  (>=0.80 -> {m['H2_pass']})")
print(f"   hallucination rate   : {m['hallucination_rate']}")
print(f"   over-refusal rate    : {m['over_refusal_rate']}")
print('='*56)

### Persona (H3) — PersonaScore harness, R0 (no refresh) and R1 (refresh @15/30)

20 scripts × 8 probe turns × 4 dims = 640 paired probes per condition (Sonnet 4.5 judge, T=0). The QPM `persona_state` channel is live per turn. Per-script mean scores stream as they complete.

In [ ]:
!python evaluate.py persona --checkpoint $CKPT_DIR/sft_final.pt --condition R0 --resume
!python evaluate.py persona --checkpoint $CKPT_DIR/sft_final.pt --condition R1 --resume

from evaluate import _persona_curve
for cond in ('R0', 'R1'):
    c = _persona_curve(f'results/persona_{cond}')
    print(f"{cond}: overall {c['overall']:.2f}  T*={c['T_star']}  dims={c['per_dimension']}")

### Judge reliability (§6.5) — weighted κ gate on a 5% re-score

In [ ]:
!python evaluate.py judge-reliability --scores-dir results/persona_R0 --frac 0.05

### QPM ablation (RQ6) — QPM-on vs `--no-qpm`

Same checkpoint, persona channel disabled. Compares expressed persona/affect with vs without the QPM conditioning (does compiling the QPM into the weights shape behaviour?).

In [ ]:
!python evaluate.py persona --checkpoint $CKPT_DIR/sft_final.pt --condition R0 --no-qpm --out-dir results/ablation_noqpm --resume
from evaluate import _persona_curve
on  = _persona_curve('results/persona_R0')['overall']
off = _persona_curve('results/ablation_noqpm/persona_R0')['overall']
print(f'RQ6 QPM ablation | QPM-on R0 overall = {on:.3f} | QPM-off R0 overall = {off:.3f} | delta = {on-off:+.3f}')

---
## P6 · Analyse — H1–H4 decision table (plan §3) + figures

`evaluate.py analyse` writes `results/analysis_data.json` **and** renders the Exp 1/2/5-style figures (png/pdf/svg) to `results/`: PersonaScore per-turn curve (R0 vs R1, T\* marked), T/E/C/S dimension bars, QA metrics vs thresholds, and the QPM-on vs `--no-qpm` ablation. The next cell displays them inline.

In [ ]:
!python evaluate.py analyse --results-dir results

import json
a = json.load(open('results/analysis_data.json'))
print('='*64)
for h in ('H1', 'H2', 'H3'):
    if h in a:
        print(f'{h}: {a[h]}')
if a.get('H4'):
    print('H4:', a['H4'])
print('-'*64)
print('DECISION:', json.dumps(a.get('decision', {}), indent=2))
print('='*64)

In [ ]:
# Display the Exp 6 figures (rendered by `evaluate.py analyse`, saved png/pdf/svg to results/).
from IPython.display import Image, display
figs = ['exp6_personascore_turn_series', 'exp6_dimension_bars',
        'exp6_qa_metrics', 'exp6_qpm_ablation']
for stem in figs:
    png = f'results/{stem}.png'
    if os.path.exists(png):
        print(stem)
        display(Image(filename=png))
    else:
        print(f'(missing {png} — run the P5 eval + P6 analyse cells first)')

---
### Next

Write `EXPERIMENT_REPORT.md` from `results/analysis_data.json` (after the run). Apply the §3 decision:

- **✓✓✓** — lock ADA knowledge-agent v0; apply the H4 refresh recommendation; proceed to the next scenario (therapy — QPM re-enters as full dynamics — or design the stack language).
- **✓✓✗** — competent QA, weak persona → add persona/episodic SFT data; retrain SFT only.
- **✓✗—** — over-confident → add unanswerable negatives + Sonnet refusal data; retrain SFT.
- **✗——** — 80M from-scratch below usability → trigger the RQ5 fallback (fine-tune SmolLM2-135M on the identical data).